In [ ]:
#这个是我接近NOAI复赛时练习用的notebook，用的OneCycleLR优化器和半精度，cuDNN因为要保证复现所以没开加速。当时时间太紧加上自己的GPU性能一般训练慢所以模型架构非常粗糙，不过也可以大致体现我的水平
import contextlib
import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import f1_score
import os
import random
torch.set_float32_matmul_precision('high')                                                                                             
seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

device = "cuda" if torch.cuda.is_available() else "cpu"
AMP_ENABLED = (device == "cuda")
AMP_DTYPE = torch.float16

if AMP_ENABLED:
    try:  # torch >= 2.3 的新 API
        from torch.amp import autocast as _autocast, GradScaler as _GradScaler
        def amp_ctx():
            return _autocast(device_type="cuda", dtype=AMP_DTYPE)
        scaler = _GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
    except (ImportError, TypeError):  # 老版本 torch
        from torch.cuda.amp import autocast as _autocast, GradScaler as _GradScaler
        def amp_ctx():
            return _autocast(dtype=AMP_DTYPE)
        scaler = _GradScaler(enabled=(AMP_DTYPE == torch.float16))
else:
    def amp_ctx():
        return contextlib.nullcontext()
    scaler = None
SCRIPT_DIR = Path.cwd()
CHECKPOINT_DIR = SCRIPT_DIR / "checkpoint"

CLASSES = ["cough", "sneeze", "laughter", "cry", "dog_bark", "siren", "noise", "none"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

HIGH_VALUE = {"cough", "sneeze", "laughter", "cry"}
CLASS_WEIGHTS = {c: 2.0 if c in HIGH_VALUE else 1.0 for c in CLASSES}

# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------
class MelDataset(Dataset):
    """
    data_dir:  directory containing .npy mel files
    csv_path:  label file (filename,label), required for train/val; can be None for inference
    """
    def __init__(self, data_dir, csv_path=None, augment=False,
                 freq_mask_param=10, time_mask_param=20):
        self.augment = augment
        self.freq_mask_param = freq_mask_param
        self.time_mask_param = time_mask_param

        data_dir = Path(data_dir)

        if csv_path is not None:
            # --- Labeled: load files listed in CSV ---
            label_df = pd.read_csv(csv_path)
            # supports column names: filename / id, label
            fname_col = "filename" if "filename" in label_df.columns else "id"
            self.samples = []  # [(npy_path, label_str), ...]
            for _, row in label_df.iterrows():
                fname = row[fname_col]
                npy_path = data_dir / (fname if fname.endswith(".npy") else fname + ".npy")
                self.samples.append((npy_path, row["label"]))
        else:
            # --- Unlabeled (inference): load all .npy from directory ---
            self.samples = [(p, None) for p in sorted(data_dir.glob("*.npy"))]

        # Load all mels into memory at once
        t0 = time.time()
        self.mels = [np.load(str(p)).astype(np.float32) for p, _ in self.samples]
        logging.info(f"Loaded {len(self.mels)} mels from {data_dir} in {time.time() - t0:.1f}s")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        logS = self.mels[idx].copy()
        if self.augment:
            logS = self._augment(logS)

        if self.samples[idx][1] is not None:
            label = CLASS_TO_IDX[self.samples[idx][1]]
            return torch.from_numpy(logS), label
        else:
            # inference mode, returns dummy label
            return torch.from_numpy(logS), 0

    def _augment(self, spec):
        n_mels, t = spec.shape
        if self.freq_mask_param > 0:
            f = np.random.randint(0, self.freq_mask_param)
            f0 = np.random.randint(0, max(1, n_mels - f))
            spec[f0:f0 + f, :] = 0.0
        if self.time_mask_param > 0:
            tm = np.random.randint(0, self.time_mask_param)
            t0 = np.random.randint(0, max(1, t - tm))
            spec[:, t0:t0 + tm] = 0.0
        return spec

    def get_ids(self):
        """Return the filename (without extension) for each sample"""
        return [p.stem for p, _ in self.samples]


# ---------------------------------------------------------------------------
# Model: SimpleCNN
# ---------------------------------------------------------------------------
# 注：AMP 下若 torch.compile 的 reduce-overhead 报 CUDA graph 错误，改成 mode="default"
@torch.compile(mode="reduce-overhead")
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Dropout(0.5),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        if x.dim() == 3:
            x = x.unsqueeze(1)
        return self.fc(self.net(x).flatten(1))


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
def _wf1(y_true, y_pred):
    pc = f1_score(y_true, y_pred, average=None, labels=range(NUM_CLASSES))
    w = np.array([CLASS_WEIGHTS[c] for c in CLASSES])
    return np.sum(pc * w) / np.sum(w)


def train_one_epoch(model, loader, criterion, optimizer, dev,
                    scaler=None, scheduler=None, ctx=amp_ctx):
    model.train()
    loss_sum, ok, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with ctx():                        # 前向在半精度下进行
            out = model(x)
            loss = criterion(out, y)
        if scaler is not None and scaler.is_enabled():
            scaler.scale(loss).backward()  # 反向用放大后的 loss
            scaler.step(optimizer)         # 必要时 unscale，并跳过 inf/nan 步
            scaler.update()                # 动态调整 scale
        else:
            loss.backward()
            optimizer.step()
        if scheduler:
            scheduler.step()
        loss_sum += loss.item() * x.size(0)
        ok += (out.argmax(1) == y).sum().item()
        n += y.size(0)
    return loss_sum / n, ok / n


@torch.no_grad()
def evaluate(model, loader, dev, ctx=amp_ctx):
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        with ctx():
            out = model(x.to(dev, non_blocking=True))
        preds.extend(out.argmax(1).cpu().tolist())
        labels.extend(y.tolist())
    acc = np.mean(np.array(preds) == np.array(labels))
    return _wf1(labels, preds), acc


# ---------------------------------------------------------------------------
# Entry
# ---------------------------------------------------------------------------
def run(epochs=100, batch_size=64, lr=5e-3, seed=42, val_ratio=0.2, skip_train=False):
    train_dir = "/bohr/meltrain-r1fx/v2/train/"
    train_csv = "/bohr/meltrain-r1fx/v2/train.csv"

    torch.manual_seed(seed)
    np.random.seed(seed)
    dev = torch.device(device)
    if dev.type == "cuda":
        logging.info(f"GPU: {torch.cuda.get_device_name(0)}")
        logging.info(f"AMP: {AMP_ENABLED} | dtype: {AMP_DTYPE if AMP_ENABLED else 'float32'}")
    else:
        logging.info("CPU mode")

    if skip_train:
        model = SimpleCNN(NUM_CLASSES).to(dev)
        model.load_state_dict(torch.load(CHECKPOINT_DIR / "best_simplecnn.pth",
                                         map_location=dev, weights_only=True))
    else:
        # Load all training data, split internally into train/val
        full_ds = MelDataset(train_dir, csv_path=train_csv, augment=False)
        n = len(full_ds)
        perm = np.random.RandomState(seed).permutation(n)
        split = int(n * (1 - val_ratio))

        # Split with Subset, enable augmentation for train portion
        train_loader = DataLoader(
            Subset(MelDataset(train_dir, csv_path=train_csv, augment=True),
                   perm[:split]),num_workers=8,prefetch_factor=4,persistent_workers=True,
            batch_size=batch_size, shuffle=True, pin_memory=(device == "cuda"),drop_last=True)
        val_loader = DataLoader(
            Subset(MelDataset(train_dir, csv_path=train_csv, augment=False),
                   perm[split:]),num_workers=8,prefetch_factor=4,persistent_workers=True,
            batch_size=batch_size, pin_memory=(device == "cuda"), drop_last=True)
        logging.info(f"Train: {split} samples, Val: {n - split} samples")

        model = SimpleCNN(NUM_CLASSES).to(dev)
        logging.info(f"Params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

        criterion = nn.CrossEntropyLoss(weight=torch.tensor([CLASS_WEIGHTS[c] for c in CLASSES], device=dev))
        optimizer = optim.Adam(model.parameters(), lr=lr)
        scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=epochs * len(train_loader))

        CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        best_f1, best_ep = 0.0, 0
        t0 = time.time()

        for ep in range(epochs):
            t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, dev, scaler=scaler, scheduler=scheduler)
            v_f1, v_acc = evaluate(model, val_loader, dev)
            logging.info(f"Epoch {ep+1}/{epochs} | Loss {t_loss:.4f} Acc {t_acc:.4f} | "
                         f"Val Acc {v_acc:.4f} F1 {v_f1:.4f}")
            if v_f1 > best_f1:
                best_f1, best_ep = v_f1, ep + 1
                torch.save(model.state_dict(), CHECKPOINT_DIR / "best_simplecnn.pth")

        logging.info(f"Done in {(time.time() - t0) / 60:.1f}min. Best F1 {best_f1:.4f} @ epoch {best_ep}")
        model.load_state_dict(torch.load(CHECKPOINT_DIR / "best_simplecnn.pth",
                                         map_location=dev, weights_only=True))

    logging.info("All done.")
    return model


# Usage:
model = run()
# run(skip_train=True)


In [ ]:
import os

if os.environ.get('DATA_PATH'):
    DATA_PATH = os.environ.get("DATA_PATH") + "/"
else:
    print("Baseline is running, this warning appears because the test set cannot be accessed. This is normal.")

import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

device = "cuda" if torch.cuda.is_available() else "cpu"
SCRIPT_DIR = Path.cwd()
CHECKPOINT_DIR = SCRIPT_DIR / "checkpoint"

CLASSES = ["cough", "sneeze", "laughter", "cry", "dog_bark", "siren", "noise", "none"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

val_dir = DATA_PATH + 'validation'
test_dir = DATA_PATH + 'test'
dev = torch.device('cuda')

# 推理同样走半精度
try:
    from torch.amp import autocast as _autocast
    amp_ctx = lambda: _autocast(device_type="cuda", dtype=torch.float16)
except (ImportError, TypeError):
    from torch.cuda.amp import autocast as _autocast
    amp_ctx = lambda: _autocast(dtype=torch.float16)


# ---------------------------------------------------------------------------
# Dataset (inference only, no labels)
# ---------------------------------------------------------------------------
class MelDataset(Dataset):
    def __init__(self, data_dir):
        data_dir = Path(data_dir)
        self.samples = [(p, None) for p in sorted(data_dir.glob("*.npy"))]
        self.mels = [np.load(str(p)).astype(np.float32) for p, _ in self.samples]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return torch.from_numpy(self.mels[idx]), 0

    def get_ids(self):
        return [p.stem for p, _ in self.samples]


# ---------------------------------------------------------------------------
# Model: SimpleCNN
# ---------------------------------------------------------------------------
@torch.compile(mode="reduce-overhead")
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Dropout(0.5),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        if x.dim() == 3:
            x = x.unsqueeze(1)
        return self.fc(self.net(x).flatten(1))


# Load trained model
model = SimpleCNN(NUM_CLASSES).to(dev)
model.load_state_dict(torch.load(CHECKPOINT_DIR / "best_simplecnn.pth",
                                 map_location=dev, weights_only=True))
model.eval()


# ---------------------------------------------------------------------------
# Inference & Save JSON
# ---------------------------------------------------------------------------
@torch.no_grad()
def predict_json(model, data_dir, out_name, dev):
    ds = MelDataset(data_dir)
    loader = DataLoader(ds, batch_size=128, pin_memory=(device == "cuda"))
    model.eval()

    ids = ds.get_ids()
    preds = []
    for x, _ in loader:
        with amp_ctx():
            out = model(x.to(dev, non_blocking=True))
        preds.extend(out.argmax(1).cpu().tolist())

    result = {sid: IDX_TO_CLASS[p] for sid, p in zip(ids, preds)}
    out = SCRIPT_DIR / f"submission_{out_name}.json"
    with open(out, "w") as f:
        json.dump(result, f, indent=4)
    logging.info(f"Saved {len(result)} predictions -> {out}")


predict_json(model, val_dir, "val", dev)
predict_json(model, test_dir, "test", dev)


In [ ]:
import zipfile
import os

# Define the files to be packaged and the compressed file name.
files_to_zip = ['submission_val.json', 'submission_test.json']
zip_filename = 'submission.zip'

# Create a zip file.
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # Add files to the zip file.
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} created successfully!')
